In [ ]:
import pandas as pd
from transformers import pipeline, AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
datos = pd.read_csv("comentarios_post_reddit.csv")

# Verificar estructura
print(datos.head())

# Solo columna comentario
comentarios = datos["comentario"].dropna()

In [ ]:
classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

tokenizer = AutoTokenizer.from_pretrained(
    "nlptown/bert-base-multilingual-uncased-sentiment"
)

In [ ]:
def dividir_en_chunks(texto, max_length=50):
    tokens = tokenizer(
        texto,
        truncation=True,
        padding=False,
        max_length=max_length,
        return_overflowing_tokens=True
    )

    input_ids = tokens["input_ids"]

    chunks = [
        tokenizer.decode(chunk, skip_special_tokens=True)
        for chunk in input_ids
    ]

    return chunks


In [ ]:
resultados = []

for comentario in comentarios:
    chunks = dividir_en_chunks(str(comentario))

    for chunk in chunks:
        resultado = classifier(chunk)[0]
        resultados.append(resultado)

In [ ]:
df_resultados = pd.DataFrame(resultados)

# Convertir "5 stars" -> 5
df_resultados["estrellas"] = df_resultados["label"].str.extract(r"(\d)").astype(int)

print(df_resultados.head())

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="estrellas", data=df_resultados)

plt.title("Distribución de Sentimientos (Comentarios Reddit)")
plt.xlabel("Estrellas")
plt.ylabel("Cantidad de fragmentos")

plt.show()

##Analisis


Como vemos en la grafica la mayoria de comentarios son negativos, siendo la gran mayoria de comentarios de una estrella, esto debido a que se seleccionaron los comentarios sobre un post de odio hacia un personaje de un juego competitivo, aqui se muestra que los usuarios coinciden en su mayoria con el autor del post o algunos defendiendose a ellos mismo y a su personaje favorito

